# Lab 05 : Plan en Étoile (DBML), Console SQL DuckDB & Dimensions Conformes
**Cours :** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format :** Lab #5 (EN : Ven 27.11.2026 / FR : Lun 30.11.2026) · 3 Heures (180 min)  
**Lien Cours :** Enchaîne directement sur le **Cours 04** (*Modélisation, Schéma en Étoile Kimball & SQL*) et prépare votre équipe au **Tutorial #2** !

---

## 🕵️ Votre mission : Détective Usine en SQL natif dans la Console DuckDB
Zéro code Pandas complexe aujourd'hui ! Vous travaillez directement dans la **Console SQL visuelle de DuckDB** — avec vos tables, histogrammes de colonnes et résultats côte à côte — pour vérifier l'intuition de la direction sur le **Cas 0 (`EuroPack Beverage Bottling — Usine #1, Gdańsk`)** :
> *« Nous pensons que `LINE_A` est notre pire source de pannes, et que nos postes de nuit tournent correctement. Vérifiez avec les vraies données — puis bâtissez les fondations de notre entrepôt Gold ! »*

**Votre parcours en 4 étapes aujourd'hui :**
1. **Plan visuel (`dbdiagram.io` · 25 min) :** Réparer 4 bugs dans `schema.dbml` et comprendre comment la télémétrie Silver (15 min) devient la table de faits par poste en Gold.
2. **Charger Silver & ouvrir la Console SQL DuckDB (15 min) :** Charger les 6 Parquets Silver dans `europack_lab05.duckdb`, charger automatiquement votre **notebook SQL natif**, et explorer vos tables à la souris dans le panneau latéral.
3. **Missions SQL, Jointures & 4 Dimensions (Dans la Console DuckDB · 115 min) :** Démarrer en douceur par la **Mission 0** (`SELECT`/`WHERE`/`LIMIT`), puis `GROUP BY`, parts en `%`, jointures, CTE, Sondage Fan-Out (`×31,9` !), et créer `Dim_Line`, `Dim_Shift`, `Dim_Format` et `Dim_Date` en pur SQL !
4. **Validation finale & Export Parquet (10 min) :** Exécuter une cellule Jupyter pour vérifier vos 4 Dimensions (`✅`), exporter les Parquets Gold et fermer `europack_lab05.duckdb` pour le **Lab 06**.

---

## 0. Initialisation & outils de vérification *(Exécuter une fois)*

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )
import json
import re
import socket
import time
import uuid
import webbrowser
import duckdb
import pandas as pd

SILVER_DIR = resolve_stage_dir("silver")
GOLD_OUT = SANDBOX_DATA / "gold"
GOLD_OUT.mkdir(parents=True, exist_ok=True)
DB_PATH = GOLD_OUT / "europack_lab05.duckdb"

# Cleanly release any prior connection/UI server if this Setup cell is re-run (prevents Windows file locks!)
if "con" in globals():
    try:
        con.execute("CALL stop_ui_server();")
    except Exception:
        pass
    try:
        con.close()
    except Exception:
        pass

if DB_PATH.exists():
    try:
        DB_PATH.unlink()
    except PermissionError as exc:
        raise PermissionError(
            f"Could not overwrite {DB_PATH} because another process is using it. "
            "Close any other open Lab 05 notebook/terminal or Restart Kernel, then re-run this cell."
        ) from exc

con = duckdb.connect(str(DB_PATH))
_UI_SERVER_STARTED = False
_UI_PORT = 4213


def check_dbml(spec: str) -> None:
    """Verify all 4 modeling bugs in schema.dbml are fixed (including [not null, ref: > ...])."""
    issues = []
    required_tables = [
        "Dim_Date", "Dim_Shift", "Dim_Line", "Dim_Format",
        "Fact_Shift_Production", "Fact_Stoppage_Events", "Fact_Telemetry_15min",
    ]
    for tbl in required_tables:
        if not re.search(rf"Table\s+{tbl}\b", spec):
            issues.append(f"Missing table: {tbl}")
    if "oee_pct" in spec.lower():
        issues.append("Bug #4 still present: non-additive ratio 'oee_pct' should NOT be stored in Fact_Shift_Production!")
    if re.search(r"Table\s+Fact_Shift_Production\s*\{[^}]*line_name", spec, re.S):
        issues.append("Bug #4 still present: descriptive text 'line_name' belongs in Dim_Line, not Fact_Shift_Production!")
    if re.search(r"ref:\s*<\s*Dim_", spec):
        issues.append("Bug #3 still present: wrong Ref direction ('<'). Many facts point to ONE dimension using '[not null, ref: > Dim_...]'!")
    fk_not_null = len(re.findall(r"\[[^\]]*not\s+null[^\]]*ref:\s*>\s*Dim_|\[[^\]]*ref:\s*>\s*Dim_[^\]]*not\s+null", spec, re.I))
    ref_count = len(re.findall(r"ref:\s*>\s*Dim_", spec))
    if ref_count < 11:
        issues.append(f"Expected 11 foreign-key links '[not null, ref: > Dim_...]' (4 on Fact_Shift_Production + 3 on Fact_Stoppage_Events + 4 on Fact_Telemetry_15min), found {ref_count}.")
    elif fk_not_null < 11:
        issues.append(f"dbdiagram.io requires 'not null' on mandatory '>' foreign keys: found {fk_not_null}/11 with 'not null' (use '[not null, ref: > Dim_...]').")
    if issues:
        print("❌ DBML still has modeling issue(s):")
        for msg in issues:
            print("   •", msg)
    else:
        (GOLD_OUT / "schema.dbml").write_text(spec.strip() + "\n", encoding="utf-8")
        print("✅ DBML Star Schema PASSED! All 7 tables (4 Dims + 3 Facts), 11 NOT NULL FK links, and additivity rules verified.")
        print(f"   Saved -> {GOLD_OUT / 'schema.dbml'}")


def _is_port_open(port: int = _UI_PORT) -> bool:
    """Fast IPv4 loopback check (avoids Windows IPv6 localhost timeout hangs)."""
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.25):
            return True
    except OSError:
        return False


def _find_ui_catalog() -> tuple[str, str] | None:
    """Return (database_name, schema_name) once DuckDB UI has attached its internal notebook tables."""
    try:
        rows = con.execute(
            "SELECT database_name, schema_name, table_name FROM duckdb_tables() "
            "WHERE table_name IN ('notebooks', 'notebook_versions')"
        ).fetchall()
        by_ns: dict[tuple[str, str], set[str]] = {}
        for db_n, sch_n, tbl_n in rows:
            by_ns.setdefault((db_n, sch_n), set()).add(tbl_n)
        for ns, tbls in by_ns.items():
            if {"notebooks", "notebook_versions"}.issubset(tbls):
                return ns
    except Exception:
        pass
    return None


def ensure_duckdb_ui_ready(port: int = _UI_PORT, wait_seconds: float = 25.0) -> bool:
    """Start DuckDB UI server safely (never hangs on Windows re-runs!) and wait for UI tables to initialize."""
    global _UI_SERVER_STARTED
    url = f"http://localhost:{port}"

    if not _UI_SERVER_STARTED:
        if _is_port_open(port):
            # Another process is already listening on port 4213 (which also locks ui.db on Windows!)
            if _find_ui_catalog() is not None:
                _UI_SERVER_STARTED = True
            else:
                print(
                    f"⚠️ Port {port} is ALREADY in use by another DuckDB/Jupyter process!\n"
                    "   On Windows, a second DuckDB UI cannot share ~/.duckdb/extension_data/ui/ui.db.\n"
                    "   👉 Close any other open Lab 05 notebook kernel or DuckDB terminal, restart this kernel, and re-run!",
                    flush=True,
                )
                return False
        else:
            print(f"⏳ Starting DuckDB SQL Console server at {url} ...", flush=True)
            con.execute("CALL start_ui_server();")
            # Wait briefly for HTTP listener to accept connections
            t_start = time.monotonic()
            while time.monotonic() - t_start < 5.0:
                if _is_port_open(port):
                    break
                time.sleep(0.2)
            _UI_SERVER_STARTED = True

    # Fast path: internal UI tables are already attached (e.g., on cell re-run with FORCE_REFRESH_NOTEBOOK = True)
    if _find_ui_catalog() is not None:
        return True

    # First-time launch: the DuckDB UI web app creates & attaches _duckdb_ui when the browser opens localhost:4213.
    # Use Python's webbrowser.open() instead of CALL start_ui() so Windows Jupyter kernels NEVER hang!
    try:
        webbrowser.open(url)
    except Exception:
        pass

    print(
        f"🌐 Opened {url} in your browser — waiting up to {int(wait_seconds)}s for DuckDB UI to finish initializing...",
        flush=True,
    )
    deadline = time.monotonic() + wait_seconds
    loc = None
    while time.monotonic() < deadline:
        loc = _find_ui_catalog()
        if loc is not None:
            break
        time.sleep(0.5)

    if loc is None:
        return False

    # Give the frontend ~1.5s to finish creating its default empty notebook before we insert & activate ours
    ui_cat, ui_sch = loc
    settle_deadline = time.monotonic() + 2.0
    while time.monotonic() < settle_deadline:
        try:
            cnt = con.execute(f'SELECT COUNT(*) FROM "{ui_cat}"."{ui_sch}".notebook_versions').fetchone()[0]
            if cnt and cnt > 0:
                time.sleep(0.5)
                break
        except Exception:
            pass
        time.sleep(0.25)
    return True


def seed_duckdb_ui_notebook(
    title: str,
    sql_cells: list[dict | str],
    db_name: str = "europack_lab05",
    force_refresh: bool = False,
    make_active: bool = True,
) -> bool:
    """Load a pre-built native SQL notebook into DuckDB's local SQL Console (idempotent!)."""
    try:
        loc = _find_ui_catalog()
        if not loc:
            return False
        ui_cat, ui_sch = loc
        tbl_nb = f'"{ui_cat}"."{ui_sch}".notebooks'
        tbl_ver = f'"{ui_cat}"."{ui_sch}".notebook_versions'
        tbl_cur = f'"{ui_cat}"."{ui_sch}".current_notebook_id'

        existing = con.execute(
            f"SELECT notebook_id, version FROM {tbl_ver} WHERE title = ? AND expires IS NULL ORDER BY version DESC LIMIT 1",
            [title],
        ).fetchone()

        def _build_payload(ver: int) -> str:
            cells_out = []
            for i, item in enumerate(sql_cells, start=1):
                q = item["query"] if isinstance(item, dict) else str(item)
                cells_out.append(
                    {
                        "query": q,
                        "useDatabase": db_name,
                        "cellId": i,
                        "isActive": (i == 1),
                        "runMode": "default",
                    }
                )
            return json.dumps(
                {
                    "cells": cells_out,
                    "currentDatabase": db_name,
                    "viewMode": {"mode": "default"},
                    "version": ver,
                    "notebookSerializationFormat": 3,
                }
            )

        def _activate(nid: str) -> None:
            if make_active:
                try:
                    con.execute(f"DELETE FROM {tbl_cur}")
                    con.execute(f"INSERT INTO {tbl_cur} VALUES (?)", [nid])
                except Exception:
                    pass

        if existing and not force_refresh:
            nb_id, ver = existing
            if ver == 1:
                con.execute(
                    f"UPDATE {tbl_ver} SET json = ? WHERE notebook_id = ? AND version = 1",
                    [_build_payload(1), nb_id],
                )
                _activate(nb_id)
                print(f"🔄 Refreshed unedited DuckDB Console notebook '{title}' ({len(sql_cells)} cells)!")
                return True
            _activate(nb_id)
            print(f"📖 DuckDB Console notebook '{title}' already exists (preserving your edits — set FORCE_REFRESH_NOTEBOOK = True to reset!).")
            return True

        if existing and force_refresh:
            nb_id, _ = existing
            max_ver = con.execute(
                f"SELECT COALESCE(MAX(version), 0) FROM {tbl_ver} WHERE notebook_id = ?",
                [nb_id],
            ).fetchone()[0]
            new_ver = int(max_ver) + 1
            con.execute(
                f"UPDATE {tbl_ver} SET expires = now() WHERE notebook_id = ? AND expires IS NULL",
                [nb_id],
            )
            con.execute(
                f"INSERT INTO {tbl_ver} (notebook_id, version, title, json, created, expires) VALUES (?, ?, ?, ?, now(), NULL)",
                [nb_id, new_ver, title, _build_payload(new_ver)],
            )
            _activate(nb_id)
            print(f"🔄 Reset DuckDB Console notebook '{title}' -> version {new_ver} ({len(sql_cells)} cells)!")
            return True

        nb_id = str(uuid.uuid4())
        nb_name = f"notebook_{nb_id[:12]}"
        con.execute(f"INSERT INTO {tbl_nb} (id, name, created) VALUES (?, ?, now())", [nb_id, nb_name])
        con.execute(
            f"INSERT INTO {tbl_ver} (notebook_id, version, title, json, created, expires) VALUES (?, 1, ?, ?, now(), NULL)",
            [nb_id, title, _build_payload(1)],
        )
        _activate(nb_id)
        print(f"✅ Pre-loaded DuckDB Console notebook '{title}' ({len(sql_cells)} cells)!")
        return True
    except Exception as exc:
        print(f"ℹ️ Could not auto-seed DuckDB Console notebook ({exc}). Use the backup .sql worksheet file instead!")
        return False


def verify_and_export_dimensions() -> bool:
    """Verify all 4 Conformed Dimensions built inside DuckDB Console and export Gold Parquet files."""
    existing = set(
        con.execute("SELECT table_name FROM information_schema.tables WHERE table_type='BASE TABLE'").df()["table_name"]
    )
    required = ("Dim_Date", "Dim_Shift", "Dim_Line", "Dim_Format")
    missing = [t for t in required if t not in existing]
    if missing:
        print(f"💡 Still waiting for Dimension table(s) to be created in DuckDB Console: {missing}")
        print("   Complete Gold Build 1–3 in the DuckDB SQL Console, then re-run this cell!")
        return False

    counts = {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in required}
    wk = con.execute("SELECT COUNT(*) FROM Dim_Date WHERE is_weekend = TRUE").fetchone()[0]
    s_min = con.execute("SELECT MIN(planned_duration_min) FROM Dim_Shift").fetchone()[0]
    expected = {"Dim_Date": 60, "Dim_Shift": 3, "Dim_Line": 4, "Dim_Format": 3}

    if counts != expected or wk != 18 or s_min != 480:
        print(f"❌ Dimension check mismatch -> counts={counts} (expected {expected}), weekend_days={wk} (expected 18), shift_min={s_min} (expected 480).")
        return False

    exported = []
    for tbl in required:
        out_p = GOLD_OUT / f"{tbl.lower()}.parquet"
        con.execute(f"COPY {tbl} TO '{out_p.as_posix()}' (FORMAT PARQUET, COMPRESSION SNAPPY)")
        exported.append(out_p.name)

    print(f"✅ ALL 4 CONFORMED DIMENSIONS VERIFIED & EXPORTED TO {GOLD_OUT}:")
    print("   Parquet files:", ", ".join(exported))
    print(con.execute("""
        SELECT 'Dim_Date' AS dimension_table, COUNT(*) AS row_count FROM Dim_Date
        UNION ALL SELECT 'Dim_Shift',  COUNT(*) FROM Dim_Shift
        UNION ALL SELECT 'Dim_Line',   COUNT(*) FROM Dim_Line
        UNION ALL SELECT 'Dim_Format', COUNT(*) FROM Dim_Format
    """).df().to_string(index=False))
    return True


print("Silver Source :", SILVER_DIR)
print("DuckDB Target :", DB_PATH)
print("DuckDB Version:", duckdb.__version__)


## 1. Plan Visuel : Réparer le Schéma en Étoile (`dbdiagram.io` · 25 min)
### Pourquoi un DBML avant le SQL ? (Et pourquoi la télémétrie 15 min devient la Production par Poste !)
* **DBML = Le plan CAO de votre entrepôt :** Comme on n'usine jamais une pièce sans plan CAO, `schema.dbml` permet de détecter tables manquantes, flèches inversées ou ratios non additifs en **2 minutes** — et chaque bloc `Table` se traduit 1 pour 1 en table DuckDB !
* **Principe Kimball — Le confort analytique prime sur la non-redondance :** En Gold (Lab 06), nous gardons volontairement **À LA FOIS** une table de Faits atomique (**`Fact_Telemetry_15min`**, `23 000` lignes — chaque relevé 15 min nettoyé !) **ET** une table de Faits pré-agrégée (**`Fact_Shift_Production`**, `1 010` lignes — OEE poste proratisé `480 min` & moyennes pondérées via `telemetry_readings`), plus **`Fact_Stoppage_Events`** (`289` lignes). Ainsi, **la BI lit uniquement Gold — jamais Silver !**
* **Pourquoi `Dim_Date` (entier `AAAAMMJJ`) & clés `NOT NULL` ?** `Dim_Date` offre des séries sans trous, des filtres `is_weekend`/`quarter` prêts à l'emploi et un calendrier unique pour *les deux* tables de Faits. Une clé entière `20261001` est lisible, triable, rapide, sans bug de fuseau horaire, et gère les dates inconnues sans `NULL`. Chaque clé étrangère porte **`[not null, ref: > Dim_...]`** !

### Activité en binôme : Corrigez les 4 bugs sur [dbdiagram.io/d](https://dbdiagram.io/d)
1. Ajoutez **`Dim_Date`** (`date_key [pk]`, `calendar_date`, `day_of_week`, `day_of_month`, `month_name`, `quarter`, `year`, `is_weekend`).
2. Ajoutez les clés étrangères (`date_key`, `shift_key`, `format_key`) dans **`Fact_Shift_Production`** avec `[not null, ref: > Dim_...]`.
3. Corrigez la flèche inversée (`<` → `>`) sur `Fact_Stoppage_Events.line_key` et ajoutez `date_key` et `shift_key`.
4. Supprimez les violations Kimball (`line_name` et le ratio non additif `oee_pct`) de **`Fact_Shift_Production`**.

<details><summary>💡 Bloqué sur la syntaxe DBML ? Cliquez pour un exemple</summary>

```dbml
Table Dim_Date {
  date_key integer [pk]
  calendar_date date
  day_of_week varchar
  day_of_month integer
  month_name varchar
  quarter integer
  year integer
  is_weekend boolean
}
// Clés étrangères dans Fact_Shift_Production (notez 'not null' !) :
  date_key integer [not null, ref: > Dim_Date.date_key]
  shift_key integer [not null, ref: > Dim_Shift.shift_key]
  line_key integer [not null, ref: > Dim_Line.line_key]
  format_key integer [not null, ref: > Dim_Format.format_key]
```
</details>

In [ ]:
# Collez ce DBML CASSÉ dans https://dbdiagram.io/d et corrigez les 4 bugs en binôme,
# puis collez votre DBML propre dans dbml_spec ci-dessous et exécutez cette cellule !
#
# LES 4 BUGS DE MODÉLISATION À CORRIGER :
#   Bug #1 : Dim_Date manque ! Ajoutez Table Dim_Date (date_key [pk], calendar_date, day_of_week,
#            day_of_month, month_name, quarter, year, is_weekend).
#   Bug #2 : Il manque date_key, shift_key et format_key dans Fact_Shift_Production.
#            (dbdiagram.io exige '[not null, ref: > Dim_...]' pour les clés étrangères obligatoires '>' !)
#   Bug #3 : Fact_Stoppage_Events a une flèche INVERSÉE ('<' au lieu de '>') sur line_key,
#            et il manque date_key et shift_key.
#   Bug #4 : Fact_Shift_Production viole Kimball en stockant du texte ('line_name')
#            ET un ratio non additif ('oee_pct') ! Supprimez ces 2 colonnes de la table de Faits.

dbml_spec = '''// EuroPack Beverage Bottling — SCHÉMA CASSÉ (Corrigez les 4 bugs sur dbdiagram.io !)
Table Dim_Shift {
  shift_key integer [pk]
  shift_code varchar [unique]
  shift_name varchar
  start_time varchar
  end_time varchar
  planned_duration_min integer
}

Table Dim_Line {
  line_key integer [pk]
  line_code varchar [unique]
  line_name varchar
  filler_type varchar
  nominal_cycle_time_sec double
  hourly_operating_cost_eur double
}

Table Dim_Format {
  format_key integer [pk]
  format_code varchar [unique]
  format_name varchar
  packaging_material varchar
  target_hourly_output integer
  unit_reject_cost_eur double
}

Table Fact_Shift_Production {
  fact_id integer [pk]
  line_key integer [not null, ref: > Dim_Line.line_key]
  // À FAIRE (Bug #2) : Ajouter date_key, shift_key, format_key avec [not null, ref: > Dim_...]
  line_name varchar          // <-- Bug #4 : Texte descriptif -> appartient à Dim_Line !
  oee_pct double             // <-- Bug #4 : Ratio non additif -> calculé en vue SQL, jamais stocké !
  planned_time_min double
  unplanned_downtime_min double
  operating_time_min double
  bottled_units bigint
  good_units bigint
  rejected_units bigint
  telemetry_readings integer
  avg_pressure_bar double
  avg_pasteurizer_temp_C double
  avg_delta_temp_C double
  avg_valve_cycles_k double
  max_valve_cycles_k double
  unplanned_downtime_cost_eur double
  reject_material_cost_eur double
}

Table Fact_Stoppage_Events {
  log_key integer [pk]
  log_id varchar [unique]
  line_key integer [not null, ref: < Dim_Line.line_key]  // <-- Bug #3 : Flèche inversée !
  // À FAIRE (Bug #3) : Ajouter date_key et shift_key avec [not null, ref: > Dim_...]
  event_timestamp timestamp
  stoppage_type varchar
  root_cause varchar
  technician_team varchar
  downtime_min double
  downtime_cost_eur double
  pre_stop_max_valve_cycles_k double
  pre_stop_max_delta_temp_C double
  pre_stop_max_pressure_bar double
  is_valve_wear_zone boolean
}

// Atomic 15-min Telemetry Fact (already properly linked — use as a syntax example!)
// Kimball lesson: Keeping BOTH Fact_Telemetry_15min (atomic) AND Fact_Shift_Production
// (aggregated) gives full analytical power in Gold without ever reaching back to Silver!
Table Fact_Telemetry_15min {
  telemetry_key integer [pk]
  date_key integer [not null, ref: > Dim_Date.date_key]
  shift_key integer [not null, ref: > Dim_Shift.shift_key]
  line_key integer [not null, ref: > Dim_Line.line_key]
  format_key integer [not null, ref: > Dim_Format.format_key]
  event_timestamp timestamp
  ambient_temp_C double
  pasteurizer_temp_C double
  delta_temp_C double
  conveyor_speed_bpm bigint
  filler_pressure_bar double
  valve_cycles_k bigint
  bottled_units bigint
  rejected_units bigint
  is_imputed_flag boolean
}
'''

check_dbml(dbml_spec)


## 2. Charger Silver & Ouvrir la Console SQL DuckDB (15 min)
Exécutez la cellule ci-dessous une fois : elle charge les 6 Parquets Silver dans `europack_lab05.duckdb`, lance la **Console SQL DuckDB (`http://localhost:4213`)**, et précharge **deux notebooks** : votre cahier principal **`Lab 05 — Missions SQL EuroPack`** (actif) et un notebook séparé **`Lab 05 — 💡 Indices & Solutions`** dans le panneau latéral (à ouvrir uniquement si bloqué !) !

### Vos 5 premières minutes dans la Console DuckDB (Sans écrire de SQL !)
Dans l'**explorateur de base à gauche** (`europack_lab05 → main`), cliquez sur vos 6 tables staging :
* **`stg_lines`** (`4` lignes) — Lignes d'embouteillage, temps de cycle & coût horaire d'arrêt (`€/h`).
* **`stg_shifts`** (`3` lignes) — `S1`, `S2`, `S3` (`8` heures = `480` min).
* **`stg_formats`** (`3` lignes) — `PET_05`, `PET_15`, `GLASS_075` & coût unitaire de rejet.
* **`stg_stoppages`** (`289` lignes) — Journaux d'arrêts maintenance (*observez les histogrammes automatiques !*).
* **`stg_telemetry`** (`23 000` lignes, grain 15 min) vs **`stg_shift_integrated`** (`1 010` lignes, grain poste-format).

In [ ]:
# 1. Materialize all 6 clean Silver Parquet files as portable DuckDB tables (stg_*)
for tbl, fname in [
    ("stg_telemetry", "silver_telemetry.parquet"),
    ("stg_stoppages", "silver_stoppages.parquet"),
    ("stg_lines", "silver_lines.parquet"),
    ("stg_shifts", "silver_shifts.parquet"),
    ("stg_formats", "silver_bottle_formats.parquet"),
    ("stg_shift_integrated", "silver_shift_integrated.parquet"),
]:
    p = SILVER_DIR / fname
    if p.exists():
        con.execute(f"CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM read_parquet('{p.as_posix()}')")

print("✅ Loaded 6 staging tables into europack_lab05.duckdb:")
print(con.execute("SHOW TABLES").df().to_string(index=False))

# 2. Launch DuckDB's Native SQL Console & Pre-Load TWO Notebooks:
#      • Active notebook  : 'Lab 05 — Missions SQL EuroPack' (clean exercises — zero spoilers!)
#      • Sidebar notebook : 'Lab 05 — 💡 Indices & Solutions' (open ONLY if you get stuck!)
#    Tip: Set FORCE_REFRESH_NOTEBOOK = True only if you want to reset your notebooks to a fresh copy.
LAUNCH_CONSOLE = True
FORCE_REFRESH_NOTEBOOK = False
SQL_NOTEBOOK_TITLE = 'Lab 05 — Missions SQL EuroPack'
HINTS_NOTEBOOK_TITLE = 'Lab 05 — 💡 Indices & Solutions'
SQL_MISSION_CELLS = ['-- =========================================================================\n'
 '-- MISSION 0 (COMMENCEZ ICI — Votre toute première requête SQL ! Sans GROUP BY !)\n'
 '-- Objectif métier : Avant de calculer des KPIs, lire le catalogue des lignes\n'
 "--                   d'embouteillage et afficher 5 arrêts bruts sur LINE_B.\n"
 "-- Astuce Console : Surlignez N'IMPORTE QUELLE requête ci-dessous et appuyez sur\n"
 '--                  Cmd+Shift+Entrée (Mac) / Ctrl+Shift+Entrée (Windows/Linux) pour exécuter uniquement\n'
 '--                  le bloc sélectionné !\n'
 '-- =========================================================================\n'
 '\n'
 "-- 0A. Lire des colonnes précises d'une table (SELECT ... FROM ...)\n"
 'SELECT line_id, line_name, filler_type, hourly_operating_cost_eur\n'
 'FROM stg_lines;\n'
 '\n'
 '-- 0B. Filtrer les lignes (WHERE) et limiter à 5 résultats (LIMIT)\n'
 'SELECT log_id, operational_shift_date, shift_code, line_code, root_cause, downtime_min\n'
 'FROM stg_stoppages\n'
 "WHERE line_code = 'LINE_B'\n"
 'LIMIT 5;',
 '-- =========================================================================\n'
 '-- MISSION 1 (DÉMO — Votre première agrégation avec GROUP BY)\n'
 "-- Objectif métier : Quel volume d'arrêt provient des pannes imprévues (Unplanned)\n"
 '--                   par rapport à la maintenance préventive (Preventive) ?\n'
 '-- =========================================================================\n'
 'SELECT stoppage_type,\n'
 '       COUNT(*)          AS incident_count,\n'
 '       SUM(downtime_min) AS total_downtime_min\n'
 'FROM stg_stoppages\n'
 'GROUP BY stoppage_type\n'
 'ORDER BY total_downtime_min DESC;',
 '-- =========================================================================\n'
 '-- MISSION 2 (WHERE + ORDER BY + LIMIT) : Les 5 plus longues pannes imprévues\n'
 '-- Objectif métier : La maintenance veut analyser en priorité les 5 plus longs\n'
 '--                   arrêts imprévus pour comprendre nos pires délais de réparation.\n'
 '-- Consigne : Complétez les 3 trous (___) et exécutez !\n'
 '-- =========================================================================\n'
 'SELECT log_id, line_code, root_cause, downtime_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = '___'\n"
 'ORDER BY ___ DESC, log_id\n'
 'LIMIT ___;',
 '-- =========================================================================\n'
 '-- MISSION 3 (GROUP BY + SUM + AVG) : LINE_A est-elle vraiment la pire machine ?\n'
 '-- Objectif métier : La direction soupçonne LINE_A. Classez chaque ligne par\n'
 "--                   cumul d'arrêts imprévus (total_min) et temps moyen de\n"
 '--                   réparation (mttr_min) pour vérifier son hypothèse !\n'
 '-- =========================================================================\n'
 'SELECT line_code,\n'
 '       COUNT(*)                    AS stoppages,\n'
 '       SUM(___)                    AS total_min,\n'
 '       ROUND(AVG(downtime_min), 2) AS mttr_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = 'Unplanned'\n"
 'GROUP BY ___\n'
 'ORDER BY ___ DESC;',
 '-- =========================================================================\n'
 '-- MISSION 4A (GROUP BY + HAVING) : Isoler les pannes chroniques (> 20 arrêts)\n'
 '-- Objectif métier : Avec un budget maintenance limité, isolez UNIQUEMENT les\n'
 '--                   causes ayant provoqué PLUS DE 20 arrêts imprévus.\n'
 '-- Rappel : WHERE filtre les lignes AVANT regroupement ; HAVING filtre les groupes APRÈS !\n'
 '-- =========================================================================\n'
 'SELECT root_cause,\n'
 '       ___ AS incident_count,\n'
 '       ___ AS total_min\n'
 'FROM stg_stoppages\n'
 'WHERE ___\n'
 'GROUP BY ___\n'
 'HAVING ___\n'
 'ORDER BY total_min DESC;',
 '-- =========================================================================\n'
 '-- MISSION 4B (Part en %) : Quel % des arrêts provient de chaque cause ?\n'
 '-- Objectif métier : La responsable maintenance veut le poids en % (Pareto) de\n'
 "--                   chacune des 4 causes d'arrêts imprévus pour justifier un\n"
 '--                   budget sur les 2 causes majeures.\n'
 '-- Nouveau motif SQL : SUM(SUM(downtime_min)) OVER () calcule le total global\n'
 '--                     sur tous les groupes afin de diviser proprement !\n'
 '-- =========================================================================\n'
 'SELECT root_cause,\n'
 '       COUNT(*)          AS incident_count,\n'
 '       SUM(downtime_min) AS total_min,\n'
 '       ROUND(SUM(downtime_min) * 100.0 / SUM(SUM(___)) OVER (), 2) AS downtime_share_pct\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = '___'\n"
 'GROUP BY root_cause\n'
 'ORDER BY total_min DESC;',
 '-- =========================================================================\n'
 '-- RAPPEL DES TYPES DE JOIN (Lisez & exécutez avant les Missions 5–6 !)\n'
 '--   1. INNER JOIN      -> Garde UNIQUEMENT les correspondances des 2 côtés (A ∩ B).\n'
 '--   2. LEFT JOIN       -> Garde TOUTES les lignes de GAUCHE ; met NULL à droite\n'
 "--                         s'il n'y a pas de match (utilisez COALESCE(..., 0) !).\n"
 '--   3. RIGHT JOIN      -> Symétrique du LEFT JOIN (rare : mettez la table principale à gauche !).\n'
 '--   4. FULL OUTER JOIN -> Garde TOUTES les lignes des 2 tables (utile en audit DQ).\n'
 '--\n'
 "-- Mini-Démo : Comparez INNER vs LEFT JOIN pour 'Capper Sensor Drift' sur nos 4 lignes !\n"
 "--             (Surlignez l'une ou l'autre requête + Cmd/Ctrl+Shift+Entrée !)\n"
 '-- =========================================================================\n'
 '\n'
 "-- INNER JOIN : les lignes sans incident 'Capper Sensor Drift' DISPARAISSENT !\n"
 'SELECT l.line_id, COUNT(s.log_id) AS capper_stops\n'
 'FROM stg_lines l\n'
 'INNER JOIN stg_stoppages s\n'
 "    ON l.line_id = s.line_code AND s.root_cause = 'Capper Sensor Drift'\n"
 'GROUP BY l.line_id\n'
 'ORDER BY l.line_id;\n'
 '\n'
 "-- LEFT JOIN : Les 4 lignes survivent toutes (0 affiché s'il n'y a eu aucun incident !)\n"
 'SELECT l.line_id, COUNT(s.log_id) AS capper_stops\n'
 'FROM stg_lines l\n'
 'LEFT JOIN stg_stoppages s\n'
 "    ON l.line_id = s.line_code AND s.root_cause = 'Capper Sensor Drift'\n"
 'GROUP BY l.line_id\n'
 'ORDER BY l.line_id;',
 '-- =========================================================================\n'
 "-- MISSION 5 (INNER JOIN) : Convertir les minutes d'arrêt en Euros (€)\n"
 "-- Objectif métier : Le directeur d'usine arbitre en Euros, pas en minutes !\n"
 '--                   Enrichissez stg_stoppages (s) avec le nom, la technologie et\n'
 '--                   hourly_operating_cost_eur depuis stg_lines (l).\n'
 '-- =========================================================================\n'
 'SELECT s.line_code,\n'
 '       l.line_name,\n'
 '       l.filler_type,\n'
 '       COUNT(*)                                       AS stoppages,\n'
 '       SUM(s.downtime_min)                            AS total_min,\n'
 '       ROUND(SUM((s.downtime_min / 60.0) * l.___), 2) AS downtime_cost_eur\n'
 'FROM stg_stoppages s\n'
 'INNER JOIN ___ l\n'
 '    ON s.line_code = l.___\n'
 "WHERE s.stoppage_type = 'Unplanned'\n"
 'GROUP BY s.line_code, l.line_name, l.filler_type\n'
 'ORDER BY downtime_cost_eur DESC;',
 '-- =========================================================================\n'
 '-- MISSION 6A (Briques de base & Intro aux CTE — Surlignez & exécutez chaque bloc !)\n'
 "-- Objectif métier : Le directeur d'usine demande :\n"
 '--   « Sur 60 jours, quel % de nos postes planifiés ont tourné SANS AUCUNE PANNE ? »\n'
 '--\n'
 '-- Pour répondre proprement, nous préparons DEUX briques avant de les joindre :\n'
 '--   Bloc 1 : Tous les (date, poste, ligne) planifiés -> 720 lignes-postes.\n'
 "--   Bloc 2 : Cumul d'arrêt imprévu par (date, poste, ligne) -> 210 postes seulement !\n"
 '--\n'
 '-- Astuce : Surlignez le Bloc 1 + Cmd/Ctrl+Shift+Entrée, puis le Bloc 2 + Cmd/Ctrl+Shift+Entrée !\n'
 '--          En Mission 6B, nous réunissons ces 2 blocs dans une CTE (WITH nom AS (...)),\n'
 '--          comme deux feuilles temporaires nommées pour cette requête !\n'
 '-- =========================================================================\n'
 '\n'
 '-- Bloc 1 (all_line_shifts) : Combien de lignes-postes uniques sur 60 jours ? (720 lignes !)\n'
 'SELECT DISTINCT operational_shift_date, shift_code, line_id\n'
 'FROM stg_telemetry\n'
 'ORDER BY 1, 2, 3;\n'
 '\n'
 '-- Bloc 2 (shift_stops) : Quelles lignes-postes ont subi une panne imprévue ? (210 lignes !)\n'
 'SELECT operational_shift_date, shift_code, line_code,\n'
 '       SUM(downtime_min) AS unplanned_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = 'Unplanned'\n"
 'GROUP BY operational_shift_date, shift_code, line_code\n'
 'ORDER BY 1, 2, 3;',
 '-- =========================================================================\n'
 '-- MISSION 6B (CTE + LEFT JOIN + COALESCE + % de Postes Sans Panne !)\n'
 '-- Objectif métier : Relier le Bloc 1 (720 postes) et le Bloc 2 (210 postes en\n'
 '--                   panne) via WITH (CTE) + LEFT JOIN sans effacer les 510 postes\n'
 '--                   sains, et calculer pct_clean_shifts !\n'
 '-- Astuce : Vous pouvez toujours surligner un SELECT interne dans WITH + Cmd/Ctrl+Shift+Entrée !\n'
 '-- =========================================================================\n'
 'WITH all_line_shifts AS (\n'
 '    SELECT DISTINCT operational_shift_date, shift_code, line_id\n'
 '    FROM stg_telemetry\n'
 '),\n'
 'shift_stops AS (\n'
 '    SELECT operational_shift_date, shift_code, line_code,\n'
 '           SUM(downtime_min) AS unplanned_min\n'
 '    FROM stg_stoppages\n'
 "    WHERE stoppage_type = 'Unplanned'\n"
 '    GROUP BY operational_shift_date, shift_code, line_code\n'
 ')\n'
 'SELECT COUNT(*)                                                          AS total_line_shifts,\n'
 '       COUNT(st.unplanned_min)                                           AS shifts_with_stoppage,\n'
 '       COUNT(*) - COUNT(st.unplanned_min)                                AS clean_shifts,\n'
 '       ROUND((COUNT(*) - COUNT(st.unplanned_min)) * 100.0 / COUNT(*), 2) AS pct_clean_shifts,\n'
 '       SUM(___(st.unplanned_min, 0))                                     AS total_unplanned_min\n'
 'FROM all_line_shifts ls\n'
 '___ JOIN shift_stops st\n'
 '    ON ls.operational_shift_date = st.operational_shift_date\n'
 '   AND ls.shift_code = st.shift_code\n'
 '   AND ls.line_id = st.line_code;',
 '-- =========================================================================\n'
 "-- SONDAGE EN SALLE (DÉMO — STOP & VOTEZ AVANT D'EXÉCUTER !) :\n"
 "-- Danger métier : Que se passe-t-il si l'on saute la modélisation Kimball et\n"
 "--                 qu'on fait un JOIN naïf entre la télémétrie 15 min (23 000 lignes)\n"
 '--                 et stg_stoppages ? (Rappel : vrai total imprévu = 22 260 min !)\n'
 '-- =========================================================================\n'
 'SELECT COUNT(*)                                 AS joined_rows,\n'
 '       SUM(st.downtime_min)                     AS FAKE_inflated_downtime_min,\n'
 '       ROUND(SUM(st.downtime_min) / 22260.0, 1) AS inflation_multiplier\n'
 'FROM stg_telemetry t\n'
 'JOIN stg_stoppages st\n'
 '  ON t.operational_shift_date = st.operational_shift_date\n'
 ' AND t.shift_code = st.shift_code\n'
 ' AND t.line_id = st.line_code\n'
 "WHERE st.stoppage_type = 'Unplanned';",
 '-- =========================================================================\n'
 '-- CONSTRUCTION GOLD 1 (DÉMO — Exécutez !) : Créer votre 1re Dimension (Dim_Line)\n'
 '-- Objectif métier : Traduire votre plan schema.dbml en tables Gold physiques\n'
 '--                   dotées de clés entières de substitution (1, 2, 3, 4).\n'
 "-- Après exécution, cliquez Rafraîchir dans l'explorateur à gauche -> Dim_Line !\n"
 '-- =========================================================================\n'
 'CREATE OR REPLACE TABLE Dim_Line AS\n'
 'SELECT ROW_NUMBER() OVER (ORDER BY line_id) AS line_key,\n'
 '       line_id                              AS line_code,\n'
 '       line_name,\n'
 '       filler_type,\n'
 '       nominal_cycle_time_sec,\n'
 '       hourly_operating_cost_eur\n'
 'FROM stg_lines;\n'
 '\n'
 'SELECT * FROM Dim_Line ORDER BY line_key;',
 '-- =========================================================================\n'
 '-- CONSTRUCTION GOLD 2 (Complétez les trous ___) : Créer Dim_Shift & Dim_Format\n'
 '-- Objectif métier : Matérialiser les Dimensions Poste (480 min) et Format\n'
 '--                   conformément à votre schéma schema.dbml.\n'
 '-- =========================================================================\n'
 'CREATE OR REPLACE TABLE Dim_Shift AS\n'
 'SELECT ROW_NUMBER() OVER (ORDER BY shift_id) AS shift_key,\n'
 '       shift_id                              AS shift_code,\n'
 '       shift_name,\n'
 '       start_time,\n'
 '       end_time,\n'
 '       (___ * 60)::INTEGER                   AS planned_duration_min\n'
 'FROM stg_shifts;\n'
 '\n'
 'CREATE OR REPLACE TABLE Dim_Format AS\n'
 'SELECT ROW_NUMBER() OVER (ORDER BY ___) AS format_key,\n'
 '       format_code,\n'
 '       format_name,\n'
 '       packaging_material,\n'
 '       target_hourly_output,\n'
 '       unit_reject_cost_eur\n'
 'FROM ___;\n'
 '\n'
 'SELECT * FROM Dim_Shift ORDER BY shift_key;',
 '-- =========================================================================\n'
 '-- CONSTRUCTION GOLD 3 (2 trous ___) : Créer Dim_Date sur 60 jours\n'
 '-- Objectif métier : Construire un calendrier conforme partagé entre Production\n'
 '--                   et Arrêts, avec attributs is_weekend/quarter prêts à filtrer !\n'
 '-- Pourquoi un entier AAAAMMJJ (20261001) ? Lisible, triable naturellement, rapide\n'
 '-- (4 octets), zéro bug de fuseau horaire/format, et gère les dates inconnues sans NULL !\n'
 '-- =========================================================================\n'
 'CREATE OR REPLACE TABLE Dim_Date AS\n'
 'WITH bounds AS (\n'
 '    SELECT MIN(operational_shift_date)::DATE AS min_d,\n'
 '           MAX(operational_shift_date)::DATE AS max_d\n'
 '    FROM stg_telemetry\n'
 '),\n'
 'dates AS (\n'
 "    SELECT unnest(generate_series(min_d, max_d, INTERVAL '1 DAY'))::DATE AS cal_date\n"
 '    FROM bounds\n'
 ')\n'
 "SELECT strftime(cal_date, '___')::INTEGER AS date_key,\n"
 '       cal_date                           AS calendar_date,\n'
 '       dayname(cal_date)                  AS day_of_week,\n'
 '       dayofmonth(cal_date)               AS day_of_month,\n'
 '       monthname(cal_date)                AS month_name,\n'
 '       quarter(cal_date)                  AS quarter,\n'
 '       year(cal_date)                     AS year,\n'
 '       CASE WHEN dayofweek(cal_date) IN (0, 6) THEN ___ ELSE FALSE END AS is_weekend\n'
 'FROM dates;\n'
 '\n'
 'SELECT * FROM Dim_Date ORDER BY date_key LIMIT 7;',
 '-- =========================================================================\n'
 "-- SPRINT DÉTECTIVE #1 : Quel matériau d'emballage a le pire taux de rejet ?\n"
 '-- Objectif métier : En partant de nos 3 formats (Dim_Format), calculer le\n'
 '--                   volume embouteillé, les rejets et le taux reject_rate_pct !\n'
 '--\n'
 '-- Deux réflexes clés de modélisation :\n'
 '--  1. On place FROM Dim_Format EN PREMIER — cela suit la pensée métier (« Je pars\n'
 '--     de mes formats, puis je rattache leurs statistiques via LEFT JOIN ! »).\n'
 "--  2. Pourquoi joindre sur f.format_code au lieu de f.format_key aujourd'hui ?\n"
 '--     Parce que stg_telemetry est encore une table SILVER (elle ne connaît que\n'
 "--     le code ERP 'GLASS_075'). Au Lab 06, on fait cette correspondance UNE FOIS\n"
 "--     pour stocker l'entier format_key dans Fact_Shift_Production — ensuite, toutes\n"
 '--     les requêtes Gold ne joignent plus que sur des entiers !\n'
 '-- =========================================================================\n'
 'SELECT f.format_code,\n'
 '       f.packaging_material,\n'
 '       SUM(t.bottled_units)                                           AS total_bottled,\n'
 '       SUM(t.rejected_units)                                          AS total_rejected,\n'
 '       ROUND(SUM(t.rejected_units) * 100.0 / SUM(t.bottled_units), 2) AS reject_rate_pct\n'
 'FROM ___ f\n'
 'LEFT JOIN stg_telemetry t\n'
 '    ON f.___ = t.bottle_format\n'
 'GROUP BY f.format_code, f.packaging_material\n'
 'ORDER BY reject_rate_pct DESC;',
 '-- =========================================================================\n'
 '-- SPRINT DÉTECTIVE #2 & #3 : Production Week-end (depuis Dim_Date !) & Équipes\n'
 '-- Objectif métier (#2) : La cadence chute-t-elle le week-end ? Partez FROM Dim_Date\n'
 '--                        pour comparer la moyenne journalière Week-end vs Semaine !\n'
 "-- Objectif métier (#3) : Quelle équipe technique absorbe le plus d'arrêts imprévus ?\n"
 '-- =========================================================================\n'
 'WITH daily_output AS (\n'
 '    SELECT d.calendar_date,\n'
 '           d.is_weekend,\n'
 '           SUM(t.bottled_units) AS daily_bottles\n'
 '    FROM ___ d\n'
 '    LEFT JOIN stg_telemetry t\n'
 '        ON d.___ = t.operational_shift_date\n'
 '    GROUP BY d.calendar_date, d.is_weekend\n'
 ')\n'
 'SELECT is_weekend,\n'
 '       COUNT(*)                     AS day_count,\n'
 '       ROUND(AVG(daily_bottles), 0) AS avg_daily_bottles\n'
 'FROM daily_output\n'
 'GROUP BY is_weekend\n'
 'ORDER BY is_weekend;\n'
 '\n'
 '-- Sprint #3 (Surlignez & appuyez sur Cmd/Ctrl+Shift+Entrée pour exécuter !) :\n'
 'SELECT technician_team,\n'
 '       COUNT(*)          AS stoppages,\n'
 '       SUM(downtime_min) AS total_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = '___'\n"
 'GROUP BY ___\n'
 'ORDER BY total_min DESC;']
SQL_HINT_CELLS = ['-- 💡 MISSION 2 (INDICE & SOLUTION) — Top 5 plus longues pannes imprévues\n'
 '-- CIBLE : Les 5 lignes affichent downtime_min = 180 (3 heures complètes !).\n'
 "-- INDICE : stoppage_type = 'Unplanned' | ORDER BY downtime_min DESC | LIMIT 5.\n"
 '--\n'
 '-- SOLUTION :\n'
 '-- SELECT log_id, line_code, root_cause, downtime_min\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- ORDER BY downtime_min DESC, log_id\n'
 '-- LIMIT 5;',
 '-- 💡 MISSION 3 (INDICE & SOLUTION) — Arrêts & MTTR par ligne\n'
 '-- CIBLE : Surprise ! LINE_B est n°1 avec 81 pannes et 8 385 min (presque 2x LINE_A : 4 755 min !).\n'
 '-- INDICE : SUM(downtime_min) | GROUP BY line_code | ORDER BY total_min DESC.\n'
 '--\n'
 '-- SOLUTION :\n'
 '-- SELECT line_code,\n'
 '--        COUNT(*)                    AS stoppages,\n'
 '--        SUM(downtime_min)           AS total_min,\n'
 '--        ROUND(AVG(downtime_min), 2) AS mttr_min\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY line_code\n'
 '-- ORDER BY total_min DESC;',
 '-- 💡 MISSION 4A (INDICE & SOLUTION) — Causes chroniques (> 20 arrêts)\n'
 '-- CIBLE : 2 causes seulement : Filler Valve Seal Leak (106 arrêts, 12 030 min)\n'
 '--         et Pasteurizer Thermal Trip (91 arrêts, 9 660 min).\n'
 '-- INDICE : COUNT(*) AS incident_count | SUM(downtime_min) AS total_min |\n'
 "--          WHERE stoppage_type = 'Unplanned' | GROUP BY root_cause | HAVING COUNT(*) > 20.\n"
 '--\n'
 '-- SOLUTION :\n'
 '-- SELECT root_cause,\n'
 '--        COUNT(*)          AS incident_count,\n'
 '--        SUM(downtime_min) AS total_min\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY root_cause\n'
 '-- HAVING COUNT(*) > 20\n'
 '-- ORDER BY total_min DESC;',
 '-- 💡 MISSION 4B (INDICE & SOLUTION) — Part en % de chaque cause\n'
 '-- CIBLE : Filler Valve Seal Leak = 54,04 % | Pasteurizer Thermal Trip = 43,40 %\n'
 '--         (Total Top 2 = 97,44 % des arrêts imprévus ! Autres causes = 1,35 % & 1,21 %.)\n'
 "-- INDICE : SUM(SUM(downtime_min)) OVER () et stoppage_type = 'Unplanned'.\n"
 '--\n'
 '-- SOLUTION :\n'
 '-- SELECT root_cause,\n'
 '--        COUNT(*)          AS incident_count,\n'
 '--        SUM(downtime_min) AS total_min,\n'
 '--        ROUND(SUM(downtime_min) * 100.0 / SUM(SUM(downtime_min)) OVER (), 2) AS downtime_share_pct\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY root_cause\n'
 '-- ORDER BY total_min DESC;',
 '-- 💡 MISSION 5 (INDICE & SOLUTION) — INNER JOIN Coût des arrêts en Euros\n'
 '-- CIBLE : LINE_B est n°1 à 29 347,50 € (10 k€ de plus que LINE_A à 19 020,00 € !).\n'
 '-- INDICE : l.hourly_operating_cost_eur | INNER JOIN stg_lines l ON s.line_code = l.line_id.\n'
 '--\n'
 '-- SOLUTION :\n'
 '-- SELECT s.line_code,\n'
 '--        l.line_name,\n'
 '--        l.filler_type,\n'
 '--        COUNT(*)                                                             AS stoppages,\n'
 '--        SUM(s.downtime_min)                                                  AS total_min,\n'
 '--        ROUND(SUM((s.downtime_min / 60.0) * l.hourly_operating_cost_eur), 2) AS downtime_cost_eur\n'
 '-- FROM stg_stoppages s\n'
 '-- INNER JOIN stg_lines l\n'
 '--     ON s.line_code = l.line_id\n'
 "-- WHERE s.stoppage_type = 'Unplanned'\n"
 '-- GROUP BY s.line_code, l.line_name, l.filler_type\n'
 '-- ORDER BY downtime_cost_eur DESC;',
 '-- 💡 MISSION 6B (INDICE & SOLUTION) — CTE + LEFT JOIN + % Postes sans panne\n'
 '-- CIBLE : total_line_shifts = 720 | shifts_with_stoppage = 210 | clean_shifts = 510 |\n'
 '--         pct_clean_shifts = 70,83 % | total_unplanned_min = 22 260.\n'
 '--         (Testez INNER à la place de LEFT : les 510 postes sains disparaissent et pct tombe à 0 % !)\n'
 '-- INDICE : COALESCE(st.unplanned_min, 0) et LEFT JOIN shift_stops st.\n'
 '--\n'
 '-- SOLUTION :\n'
 '-- WITH all_line_shifts AS (\n'
 '--     SELECT DISTINCT operational_shift_date, shift_code, line_id\n'
 '--     FROM stg_telemetry\n'
 '-- ),\n'
 '-- shift_stops AS (\n'
 '--     SELECT operational_shift_date, shift_code, line_code,\n'
 '--            SUM(downtime_min) AS unplanned_min\n'
 '--     FROM stg_stoppages\n'
 "--     WHERE stoppage_type = 'Unplanned'\n"
 '--     GROUP BY operational_shift_date, shift_code, line_code\n'
 '-- )\n'
 '-- SELECT COUNT(*)                                                          AS total_line_shifts,\n'
 '--        COUNT(st.unplanned_min)                                           AS shifts_with_stoppage,\n'
 '--        COUNT(*) - COUNT(st.unplanned_min)                                AS clean_shifts,\n'
 '--        ROUND((COUNT(*) - COUNT(st.unplanned_min)) * 100.0 / COUNT(*), 2) AS pct_clean_shifts,\n'
 '--        SUM(COALESCE(st.unplanned_min, 0))                                AS total_unplanned_min\n'
 '-- FROM all_line_shifts ls\n'
 '-- LEFT JOIN shift_stops st\n'
 '--     ON ls.operational_shift_date = st.operational_shift_date\n'
 '--    AND ls.shift_code = st.shift_code\n'
 '--    AND ls.line_id = st.line_code;',
 '-- 💡 CONSTRUCTION GOLD 2 (INDICE & SOLUTION) — Créer Dim_Shift & Dim_Format\n'
 '-- CIBLE : Dim_Shift = 3 lignes (planned_duration_min = 480) ; Dim_Format = 3 lignes.\n'
 '-- INDICE : (planned_duration_hours * 60)::INTEGER | ORDER BY format_code | FROM stg_formats.\n'
 '--\n'
 '-- SOLUTION :\n'
 '-- CREATE OR REPLACE TABLE Dim_Shift AS\n'
 '-- SELECT ROW_NUMBER() OVER (ORDER BY shift_id) AS shift_key,\n'
 '--        shift_id                              AS shift_code,\n'
 '--        shift_name, start_time, end_time,\n'
 '--        (planned_duration_hours * 60)::INTEGER AS planned_duration_min\n'
 '-- FROM stg_shifts;\n'
 '--\n'
 '-- CREATE OR REPLACE TABLE Dim_Format AS\n'
 '-- SELECT ROW_NUMBER() OVER (ORDER BY format_code) AS format_key,\n'
 '--        format_code, format_name, packaging_material,\n'
 '--        target_hourly_output, unit_reject_cost_eur\n'
 '-- FROM stg_formats;',
 '-- 💡 CONSTRUCTION GOLD 3 (INDICE & SOLUTION) — Créer Dim_Date (60 jours)\n'
 '-- CIBLE : 60 lignes (2026-10-01 .. 2026-11-29), date_key = 20261001..20261129, 18 jours de week-end.\n'
 "-- INDICE : strftime(cal_date, '%Y%m%d')::INTEGER et THEN TRUE ELSE FALSE.\n"
 '--\n'
 '-- SOLUTION :\n'
 '-- CREATE OR REPLACE TABLE Dim_Date AS\n'
 '-- WITH bounds AS (\n'
 '--     SELECT MIN(operational_shift_date)::DATE AS min_d,\n'
 '--            MAX(operational_shift_date)::DATE AS max_d\n'
 '--     FROM stg_telemetry\n'
 '-- ),\n'
 '-- dates AS (\n'
 "--     SELECT unnest(generate_series(min_d, max_d, INTERVAL '1 DAY'))::DATE AS cal_date\n"
 '--     FROM bounds\n'
 '-- )\n'
 "-- SELECT strftime(cal_date, '%Y%m%d')::INTEGER AS date_key,\n"
 '--        cal_date AS calendar_date, dayname(cal_date) AS day_of_week,\n'
 '--        dayofmonth(cal_date) AS day_of_month, monthname(cal_date) AS month_name,\n'
 '--        quarter(cal_date) AS quarter, year(cal_date) AS year,\n'
 '--        CASE WHEN dayofweek(cal_date) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend\n'
 '-- FROM dates;',
 '-- 💡 SPRINT #1 (INDICE & SOLUTION) — Taux de rejet par format\n'
 '-- CIBLE : GLASS_075 (Amber Glass 450g) = 4,29 % de rejet (plus de 2x PET_05 1,84 % / PET_15 1,87 % !).\n'
 '-- INDICE : FROM Dim_Format f LEFT JOIN stg_telemetry t ON f.format_code = t.bottle_format.\n'
 '--\n'
 '-- SOLUTION :\n'
 '-- SELECT f.format_code,\n'
 '--        f.packaging_material,\n'
 '--        SUM(t.bottled_units)                                           AS total_bottled,\n'
 '--        SUM(t.rejected_units)                                          AS total_rejected,\n'
 '--        ROUND(SUM(t.rejected_units) * 100.0 / SUM(t.bottled_units), 2) AS reject_rate_pct\n'
 '-- FROM Dim_Format f\n'
 '-- LEFT JOIN stg_telemetry t\n'
 '--     ON f.format_code = t.bottle_format\n'
 '-- GROUP BY f.format_code, f.packaging_material\n'
 '-- ORDER BY reject_rate_pct DESC;',
 '-- 💡 SPRINT #2 & #3 (INDICE & SOLUTION) — Week-end vs Semaine & Équipes\n'
 '-- CIBLE #2 : Semaine (42 jours) = ~338 449 btl/j vs Week-end (18 jours) = ~340 060 btl/j (usine 24/7 !).\n'
 '-- CIBLE #3 : Maint_Mechanical absorbe 21 690 min (97,4 %) contre 570 min pour Maint_Controls !\n'
 '-- INDICE : FROM Dim_Date d LEFT JOIN stg_telemetry t ON d.calendar_date = t.operational_shift_date ;\n'
 "--          Sprint #3 : WHERE stoppage_type = 'Unplanned' GROUP BY technician_team.\n"
 '--\n'
 '-- SOLUTION (#2) :\n'
 '-- WITH daily_output AS (\n'
 '--     SELECT d.calendar_date, d.is_weekend, SUM(t.bottled_units) AS daily_bottles\n'
 '--     FROM Dim_Date d\n'
 '--     LEFT JOIN stg_telemetry t ON d.calendar_date = t.operational_shift_date\n'
 '--     GROUP BY d.calendar_date, d.is_weekend\n'
 '-- )\n'
 '-- SELECT is_weekend, COUNT(*) AS day_count, ROUND(AVG(daily_bottles), 0) AS avg_daily_bottles\n'
 '-- FROM daily_output GROUP BY is_weekend ORDER BY is_weekend;\n'
 '--\n'
 '-- SOLUTION (#3) :\n'
 '-- SELECT technician_team, COUNT(*) AS stoppages, SUM(downtime_min) AS total_min\n'
 "-- FROM stg_stoppages WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY technician_team ORDER BY total_min DESC;']

if LAUNCH_CONSOLE:
    ui_ready = ensure_duckdb_ui_ready(port=_UI_PORT, wait_seconds=25.0)
    seeded_h = False
    seeded_m = False
    if ui_ready:
        # Seed Hints & Solutions in the sidebar first (inactive), then activate the clean Missions notebook!
        seeded_h = seed_duckdb_ui_notebook(HINTS_NOTEBOOK_TITLE, SQL_HINT_CELLS, force_refresh=FORCE_REFRESH_NOTEBOOK, make_active=False)
        seeded_m = seed_duckdb_ui_notebook(SQL_NOTEBOOK_TITLE, SQL_MISSION_CELLS, force_refresh=FORCE_REFRESH_NOTEBOOK, make_active=True)
    print(f"\n🚀 Console SQL DuckDB active sur http://localhost:{_UI_PORT}")
    if seeded_m:
        print("   👉 IMPORTANT : Appuyez sur F5 (Rafraîchir) dans l'onglet navigateur DuckDB pour afficher les notebooks !")
        print(f"   • Cahier principal (actif) : '{SQL_NOTEBOOK_TITLE}' ({len(SQL_MISSION_CELLS)} cellules)")
        print(f"   • Bloqué ? Panneau latéral : '{HINTS_NOTEBOOK_TITLE}' ({len(SQL_HINT_CELLS)} cellules indices/solutions)")
    else:
        print("   ⚠️ Le préchargement n'a pas encore pu s'achever (la Console DuckDB est encore en cours d'ouverture).")
        print(f"      1. Ouvrez http://localhost:{_UI_PORT} dans Chrome/Edge et attendez l'affichage complet de DuckDB.")
        print("      2. Relancez CETTE cellule une fois (< 1 s, sans blocage), puis appuyez sur F5 dans le navigateur !")
        print("      3. Ou utilisez le fichier SQL de secours :", "lab_05_sql_missions_fr.sql")
    print("   • Raccourci : Surlignez un bloc SQL + Cmd+Shift+Entrée (Mac) / Ctrl+Shift+Entrée (Windows/Linux) pour l'exécuter seul !")
    print("   • Fichier SQL de secours hors-ligne :", "lab_05_sql_missions_fr.sql")
else:
    print("ℹ️ Passez LAUNCH_CONSOLE = True et exécutez cette cellule pour lancer la Console SQL DuckDB (http://localhost:4213) !")


## 3. Missions SQL & Construction des Dimensions *(Dans la Console DuckDB !)*
Basculez dans la **Console SQL DuckDB** (`http://localhost:4213`) et avancez dans **`Lab 05 — Missions SQL EuroPack`** (secours : `lab_05_sql_missions_fr.sql`). En cas de blocage, ouvrez le notebook séparé **`Lab 05 — 💡 Indices & Solutions`** dans le panneau latéral — chaque cellule indique clairement la mission correspondante !

> **Astuce Console :** Surlignez *n'importe quel* bloc SQL (même un `SELECT` interne d'une CTE `WITH` !) et appuyez sur **`Cmd + Shift + Entrée` (Mac)** ou **`Ctrl + Shift + Entrée` (Windows/Linux)** pour exécuter uniquement la sélection !

| Mission | Notion pratiquée | Résultat métier clé |
| :--- | :--- | :--- |
| **Mission 0** *(Démo)* | `SELECT`, `FROM`, `WHERE`, `LIMIT` de base | Lire le catalogue lignes & aperçu des arrêts `LINE_B` |
| **Mission 1** *(Démo)* | Premier `GROUP BY` (`COUNT`, `SUM`) | `Unplanned` : `211` / `22 260 min` · `Preventive` : `78` / `3 510 min` |
| **Mission 2** | `WHERE`, `ORDER BY`, `LIMIT` | Les 5 plus longues pannes durent toutes **`180 min`** |
| **Mission 3** | `GROUP BY` + `AVG` (`MTTR`) | **`LINE_B`** est n° 1 (`81` arrêts, **`8 385 min`**) — presque 2× `LINE_A` ! |
| **Mission 4A** | `GROUP BY` + `HAVING` | Seulement **2 causes chroniques** `> 20` arrêts (`106` & `91`) |
| **Mission 4B** | Part en `%` (`OVER ()`) | Top 2 causes = **`54,04 % + 43,40 % = 97,44 %`** des arrêts imprévus ! |
| **Rappel JOIN + M5** | 4 types de `JOIN` + `INNER JOIN` (€) | **`LINE_B`** coûte **`29 347,50 €`** (`10 k€` de plus que `LINE_A` !) |
| **Mission 6A → 6B** | Briques CTE + `LEFT JOIN` + `COALESCE` | **`510 / 720`** postes sans panne (**`70,83 %`** !) |
| **🗳️ Sondage** | Explosion Fan-Out *(Votez d'abord !)* | Le join naïf gonfle les arrêts **`×31,9`** → **`710 400 min`** ! |
| **Gold Builds 1–3** | `CREATE OR REPLACE TABLE Dim_*` | Créer **`Dim_Line` (`4`)**, **`Dim_Shift` (`3`)**, **`Dim_Format` (`3`)**, **`Dim_Date` (`60`)** |
| **⚡ Sprints #1–#3** | `FROM Dim_Format` & `FROM Dim_Date` (`LEFT JOIN`) | `GLASS_075` = **`4,29 %`** rejet · Week-end ≈ Semaine · `Maint_Mechanical` = `97,4 %` |

In [ ]:
# Réalisez les Missions 0 à 6B, le Sondage Fan-Out, les Constructions Gold 1 à 3 et le Sprint
# directement dans la Console SQL DuckDB (http://localhost:4213) !
#
# Exécutez cette cellule à tout moment pour voir les tables présentes dans europack_lab05.duckdb :
print("Tables actuelles dans europack_lab05.duckdb :")
print(con.execute("SHOW TABLES").df().to_string(index=False))


## 4. Validation Finale : Vérifier vos 4 Dimensions, Exporter en Parquet & Fermer
Une fois `Dim_Line`, `Dim_Shift`, `Dim_Format` et `Dim_Date` créées dans la Console SQL DuckDB (Constructions Gold 1–3), exécutez cette cellule pour valider les volumes, exporter `dim_*.parquet`, arrêter la Console et fermer `europack_lab05.duckdb` pour le **Lab 06** !

In [ ]:
# Run this final cell once you have built Dim_Line, Dim_Shift, Dim_Format, and Dim_Date in DuckDB Console!
ok = verify_and_export_dimensions()

# Stop the DuckDB SQL Console cleanly (if it was started) and release the file lock for Lab 06
try:
    con.execute("CALL stop_ui_server();")
    _UI_SERVER_STARTED = False
except Exception:
    pass

tables = con.execute("SELECT table_name FROM information_schema.tables WHERE table_type='BASE TABLE' ORDER BY 1").df()["table_name"].tolist()
con.close()
print("\nClosed:", DB_PATH)
print("Tables saved for Lab 06:", tables)
